In [ ]:
import librosa as lb
import tensorflow as tf
from ai_edge_litert.interpreter import Interpreter

class YamnentModel:
    def __init__(self):
        self.__interpreter = Interpreter(
            model_path="yamnet_model.tflite"
        )

        self.input_details = self.__interpreter.get_input_details()
        self.output_details = self.__interpreter.get_output_details()

    def inference(self, waveforms):
        
        self.__interpreter.resize_tensor_input(
            self.input_details[0]["index"],
            [len(waveforms)]
        )

        self.__interpreter.allocate_tensors()

        self.__interpreter.set_tensor(
            self.input_details[0]['index'],
            waveforms
        )

        self.__interpreter.invoke()

        embeddings = self.__interpreter.get_tensor(
            self.output_details[1]["index"]
        )

        return embeddings
    
class GenreClassificationModel:
    def __init__(self):
        self.__interpreter = Interpreter(
            model_path="genre_classification_model.tflite"
        )

        self.input_details = self.__interpreter.get_input_details()
        self.output_details = self.__interpreter.get_output_details()

    def inference(self, waveforms):
      
        self.__interpreter.allocate_tensors()

        self.__interpreter.set_tensor(
            self.input_details[0]['index'],
            waveforms
        )

        self.__interpreter.invoke()

        embeddings = self.__interpreter.get_tensor(
            self.output_details[0]["index"]
        )

        return embeddings
    
def argmax(probabilities):
    highest_prob_index = -1
    highest_prob = probabilities[0][0]

    for i in range(0, len(probabilities)):
        for j in range(0, len(probabilities[i])):
            if probabilities[i][j] > highest_prob:
                highest_prob = probabilities[i][j]
                highest_prob_index = j

    return highest_prob_index 


In [5]:
genre_labels = [
    "blues",
    "classical",
    "country",
    "disco",
    "hiphop",
    "jazz",
    "metal",
    "pop",
    "reggae",
    "rock"
]

yamnet_model = YamnentModel()
genre_classification_model = GenreClassificationModel()

In [ ]:

waveform, _ = lb.load("your_song_path", sr=16000, mono=True)

result = yamnet_model.inference(waveforms=waveform)
preds = genre_classification_model.inference([result])

highest_prob_index = argmax(preds)

print(genre_labels[highest_prob_index])

jazz
